# Real-time Unique Person Counting (Persistence Version)

Esta versão foi ajustada para ser **mais persistente**. Se você sair da câmera e voltar rápido, o sistema tentará manter o mesmo ID.

### Step 1: Requirements
```bash
pip install ultralytics opencv-python torch torchvision ipywidgets
```

In [ ]:
import cv2
import torch
import os
import time
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display

# Evita crash comum de biblioteca duplicada no Windows
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

print("--- Setup ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    # Carrega o modelo uma única vez para economizar memória
    model = YOLO("yolo11n.pt")
    model.to(device)
    print(f"✅ Modelo carregado na {device}.")
except Exception as e:
    print(f"❌ Erro: {e}")

### Step 2: Executar com Memória de Tracking

**Dica de Teste:** Toda vez que você inicia esta célula, o contador e a memória interna do tracker são **zerados**.

In [ ]:
image_widget = widgets.Image(format='jpeg', width=640, height=480)
display(image_widget)

cap = cv2.VideoCapture(0)

# --- TOTAL RESET (LIMPEZA DE RESQUÍCIOS) ---
unique_person_ids = set() 

# Resetar os trackers internos do modelo YOLO de forma segura
if hasattr(model, 'predictor') and model.predictor is not None:
    model.predictor.trackers = None

print("🔄 Sistema e trackers resetados para novo teste.")
# -------------------------------------------

prev_time = 0

if not cap.isOpened():
    print("❌ Erro ao abrir webcam.")
else:
    print("✅ Rodando... Clique em 'Stop' no topo para parar.")
    
    try:
        while True:
            success, frame = cap.read()
            if not success: break

            # TRACKING COM PERSISTÊNCIA
            results = model.track(
                source=frame, 
                persist=True, 
                verbose=False, 
                classes=[0], 
                tracker="bytetrack.yaml",
                conf=0.3,
                iou=0.5
            )

            annotated_frame = frame.copy()
            if results and results[0].boxes.id is not None:
                ids = results[0].boxes.id.int().cpu().tolist()
                for obj_id in ids:
                    unique_person_ids.add(obj_id)
                annotated_frame = results[0].plot()

            # FPS e UI
            curr_time = time.time()
            fps = 1 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0
            prev_time = curr_time

            cv2.rectangle(annotated_frame, (0, 0), (280, 100), (0, 0, 0), -1)
            cv2.putText(annotated_frame, f"FPS: {int(fps)}", (20, 35), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(annotated_frame, f"Total Geral: {len(unique_person_ids)}", (20, 75), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

            _, buffer = cv2.imencode('.jpg', annotated_frame)
            image_widget.value = buffer.tobytes()
            
    except KeyboardInterrupt: pass
    finally:
        cap.release()
        print(f"Finalizado. Total de IDs detectados: {len(unique_person_ids)}")
        unique_person_ids.clear()